# Clean real-space BdG

Dispersion, self-consistent gap, complete signed spectrum, broadened positive-mode spectrum, and residual history.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
from src.lattice import square_lattice_hamiltonian, square_lattice_dispersion, momentum_grid
from src.gap_equation import uniform_gap
from src.solver import solve_bdg
from src.disorder import onsite_disorder
from src.hamiltonian import gaussian_dos
print("Project:", ROOT)


In [ ]:
L, t, t_prime, mu, U, T = 6, 1., 0.1, -0.5, 2.2, 0.08
k = np.linspace(-np.pi, np.pi, 100)
kx, ky = np.meshgrid(k,k,indexing="ij")
xi = square_lattice_dispersion(kx,ky,t,t_prime,mu)
plt.contourf(kx,ky,xi,35,cmap="coolwarm"); plt.colorbar(label="ξ(k)/t"); plt.xlabel("kx"); plt.ylabel("ky"); plt.show()


In [ ]:
K = square_lattice_hamiltonian(L,t,t_prime,mu)
guess = uniform_gap(L,U,T,t,t_prime,mu)
result = solve_bdg(K,U,T,initial_gap=guess,tolerance=1e-8,max_iterations=100)
print("converged:",result.converged,"iterations:",result.iterations,"mean |Δ|:",np.mean(np.abs(result.gap)))
plt.semilogy(np.maximum(result.residual_history,1e-16)); plt.xlabel("Iteration"); plt.ylabel("max update/t"); plt.show()
plt.plot(result.all_energies,".",ms=3); plt.axhline(0,color="k",lw=.7); plt.ylabel("Energy/t"); plt.xlabel("Sorted state"); plt.show()
grid=np.linspace(0,4,400); dos=gaussian_dos(result.energies,grid,.06)
plt.plot(grid,dos); plt.xlabel("Positive mode energy/t"); plt.ylabel("Normalized mode DOS"); plt.show()
